# Task 2 — Local Generative Causal LM (Qwen2.5-Coder-7B-Instruct)

**Mô hình mã nguồn mở chạy 100% Cục bộ / Kaggle GPU — Không dùng API ngoài**

1. Settings: Accelerator **GPU T4 x2**, Internet **On**.
2. Mô hình: `Qwen/Qwen2.5-Coder-7B-Instruct` (mô hình chuyên code hàng đầu thế giới phân khúc < 10B).
3. Cơ chế: **Causal Language Model (Generative In-Context Learning)** — mô hình đọc đề bài, đọc code C++ từng dòng, kết hợp compile log và ví dụ mẫu gold để tự chẩn đoán và sinh nhãn lỗi JSON.
4. Kết quả: sinh ra `/kaggle/working/task2_predictions.json` hợp lệ 100% chuẩn định dạng nộp của ban tổ chức.

In [ ]:
REPO = 'github.com/tiendinhquang2104/llm-final-project.git'
BRANCH = 'task-2-fewshot-clm'
WORK = '/kaggle/tmp'
FEW_SHOT_K = 2  # Số ví dụ mẫu gold đưa vào prompt (2 là tối ưu context và tốc độ)
MODEL_NAME = 'Qwen/Qwen2.5-Coder-7B-Instruct'

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try: return secrets.get_secret(name)
    except Exception: return ''
token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
os.makedirs(WORK, exist_ok=True)
REPO_DIR = f'{WORK}/repo'
if os.path.exists(REPO_DIR):
    print('Updating repo to latest commit on', BRANCH)
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    print('Cloning repo on branch', BRANCH)
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret('HF_TOKEN')
print('Current commit:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q 'transformers>=4.51,<5' accelerate safetensors scikit-learn pyyaml bitsandbytes
!g++ --version

In [ ]:
# Tự động kiểm tra và giải nén dữ liệu từ /kaggle/input
import os, subprocess
if not os.path.exists('data/raw/sample_dataset') or not os.path.exists('data/raw/public_test_release'):
    print('==> Auto-unpacking datasets into data/raw...')
    subprocess.run(['python', 'scripts/unpack_data.py', '/kaggle/input', '--dest', 'data/raw'], check=False)

print('Data check:')
!ls -la data/raw

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess

# Chạy suy luận Generative Causal LM với Qwen2.5-Coder-7B-Instruct
cmd = [
    'python', 'scripts/run_task2_coder.py',
    '--model', MODEL_NAME,
    '--few-shot-k', str(FEW_SHOT_K),
    '--max-new-tokens', '256'
]
print('Running command:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json, shutil
src_pred = 'outputs/task2/task2_coder_local/predictions.json'
dst_pred = '/kaggle/working/task2_predictions.json'
shutil.copy(src_pred, dst_pred)

from src.task2.evaluator import validate_predictions
sub = json.load(open(dst_pred, encoding='utf-8'))
errs = validate_predictions(sub)
print(f'==> File {dst_pred} sẵn sàng nộp!')
print(f'Số bài nộp: {len(sub)} | Lỗi định dạng: {errs}')
assert len(errs) == 0, f'Có lỗi định dạng: {errs}'